In [5]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
pip install torch_geometric rdkit


In [7]:
%cd /content/drive/MyDrive/DrugPropertyProject/TraGT/

/content/drive/MyDrive/DrugPropertyProject/TraGT


In [8]:
import torch
import numpy as np
from rdkit import Chem
from collections import defaultdict
from torch.utils.data import Dataset

In [9]:
def load_data_long(dataset, device):
    mole_dict = {1: "H", 2: "He", 3: "Li", 4: "Be", 5: "B", 6: "C", 7: "N", 8: "O", 9: "F", 10: " Ne",
                11: "Na", 12:"Mg", 13: "Al", 14:"Si", 15:"P", 16: "S", 17: "Cl", 18:"Ar", 19:"K", 20:"Ca", 22:"Ti", 24:"Cr", 26:"Fe", 28:"Ni",
                29:"Cu", 31:"Ga", 32:"Ge", 34:"Se", 35:"Br", 40:"Zr", 44:"Ru", 45:"Rh", 46:"Pd", 47:"Ag", 50:"Sn", 51:"Sb", 52:"Te", 53: "I", 65:"Tb", 75:"Re", 77:"Ir", 78:"Pt", 79:"Au", 80:"Hg",
                81:"Tl", 82:"Pb", 83:"Bi"}

    pair_list = ["Br", "Cl", "Si", "Na", "Ca", "Ge", "Cu", "Au", "Sn", "Tb", "Pt", "Re", "Ru", "Bi", "Li", "Fe", "Sb", "Hg","Pb", "Se", "Ag","Cr","Pd","Ga","Mg","Ni","Ir","Rh","Te","Ti","Al","Zr","Tl"]

    data_file = f"/content/drive/MyDrive/DrugPropertyProject/TraGT/datasets/{dataset}_train.txt"
    file = open(data_file, "r")
    node_types = set()
    label_types = set()
    tr_len = 0
    for line in file:
        tr_len += 1
        smiles = line.split("\t")[1]
        label = line.split("\t")[2][:-1]
        i = 0
        s = []
        while i < len(smiles):
            if i < len(smiles)-1 and (smiles[i] + smiles[i+1]) in pair_list:
                s.append(smiles[i] + smiles[i+1])
                i += 2
            else:
                s.append(smiles[i].upper())
                i += 1
        node_types |= set(s)
        label_types.add(label)
    file.close()

    te_len = 0
    data_file = f"/content/drive/MyDrive/DrugPropertyProject/TraGT/datasets/{dataset}_test.txt"
    file = open(data_file, "r")
    for line in file:
        te_len += 1
        smiles = line.split("\t")[1]
        label = line.split("\t")[2][:-1]
        i = 0
        s = []
        while i < len(smiles):
            if i < len(smiles)-1 and (smiles[i] + smiles[i+1]) in pair_list:
                s.append(smiles[i] + smiles[i+1])
                i += 2
            else:
                s.append(smiles[i].upper())
                i += 1
        node_types |= set(s)
        label_types.add(label)
    file.close()

    #print(tr_len)
    #print(te_len)

    node2index = {n: i for i, n in enumerate(node_types)}
    label2index = {l: i for i, l in enumerate(label_types)}

    #print(node2index)
    #print(label2index)

    data_file = f"/content/drive/MyDrive/DrugPropertyProject/TraGT/datasets/{dataset}_train.txt"
    file = open(data_file, "r")
    train_adjlists = []
    train_features = []
    train_sequence = []
    train_fg = []
    train_labels = torch.zeros(tr_len)
    for line in file:
        smiles = line.split("\t")[1]
        fg = extract_functional_groups(smiles)
        label = line.split("\t")[2][:-1]
        mol = Chem.MolFromSmiles(smiles)
        graph_nodes = []
        for atom in mol.GetAtoms():
            graph_nodes.append(mole_dict[atom.GetAtomicNum()])
        # print(graph_nodes)
        i = 0
        s = 0
        while i < len(smiles):
            if i < len(smiles)-1 and (smiles[i] + smiles[i+1]) in pair_list:
                i += 2
            else:
                i += 1
            s += 1

        feature = torch.zeros(s, len(node_types))

        map = {}
        se_num = 0
        gr_num = 0
        i = 0
        smiles_seq = []
        while i < len(smiles):
            this_str = smiles[i]
            if i < len(smiles)-1 and (smiles[i] + smiles[i+1]) in pair_list:
                this_str = smiles[i] + smiles[i+1]
                i += 2
            else:
                this_str = this_str.upper()
                i += 1
            smiles_seq.append(node2index[this_str])
            if this_str in graph_nodes and this_str == mole_dict[mol.GetAtoms()[gr_num].GetAtomicNum()]:
                map[gr_num] = se_num
                gr_num += 1
            feature[se_num, node2index[this_str]] = 1
            se_num += 1


        adj_list = defaultdict(list)
        for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()
            # print(i,j)
            typ = bond.GetBondType()
            adj_list[map[i]].append(map[j])
            adj_list[map[j]].append(map[i])
            if typ == Chem.rdchem.BondType.DOUBLE:
                adj_list[map[i]].append(map[j])
                adj_list[map[j]].append(map[i])
            elif typ == Chem.rdchem.BondType.TRIPLE:
                adj_list[map[i]].append(map[j])
                adj_list[map[j]].append(map[i])
                adj_list[map[i]].append(map[j])
                adj_list[map[j]].append(map[i])

        # train_labels[len(train_adjlists)]= int(label2index[label])
        train_labels[len(train_adjlists)]= int(label)
        train_adjlists.append(adj_list)
        train_features.append(torch.FloatTensor(feature).to(device))
        train_sequence.append(torch.tensor(smiles_seq))
        train_fg.append(fg)
    file.close()

    data_file = f"/content/drive/MyDrive/DrugPropertyProject/TraGT/datasets/{dataset}_test.txt"
    file = open(data_file, "r")
    test_adjlists = []
    test_features = []
    test_sequence = []
    test_fg = []
    test_labels = np.zeros(te_len)
    for line in file:
        smiles = line.split("\t")[1]
        # print(smiles)
        fg = extract_functional_groups(smiles)
        label = line.split("\t")[2][:-1]
        mol = Chem.MolFromSmiles(smiles)
        graph_nodes = []
        for atom in mol.GetAtoms():
            graph_nodes.append(mole_dict[atom.GetAtomicNum()])
        # print(graph_nodes)
        i = 0
        s = 0
        while i < len(smiles):
            if i < len(smiles)-1 and (smiles[i] + smiles[i+1]) in pair_list:
                i += 2
            else:
                i += 1
            s += 1

        feature = torch.zeros(s, len(node_types))

        map = {}
        se_num = 0
        gr_num = 0
        i = 0
        smiles_seq = []
        while i < len(smiles):
            this_str = smiles[i]
            if i < len(smiles)-1 and (smiles[i] + smiles[i+1]) in pair_list:
                this_str = smiles[i] + smiles[i+1]
                i += 2
            else:
                this_str = this_str.upper()
                i += 1
            smiles_seq.append(node2index[this_str])
            if this_str in graph_nodes and this_str == mole_dict[mol.GetAtoms()[gr_num].GetAtomicNum()]:
                map[gr_num] = se_num
                gr_num += 1
            feature[se_num, node2index[this_str]] = 1
            se_num += 1

        adj_list = defaultdict(list)
        for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()
            # print(i,j)
            typ = bond.GetBondType()
            adj_list[map[i]].append(map[j])
            adj_list[map[j]].append(map[i])
            if typ == Chem.rdchem.BondType.DOUBLE:
                adj_list[map[i]].append(map[j])
                adj_list[map[j]].append(map[i])
            elif typ == Chem.rdchem.BondType.TRIPLE:
                adj_list[map[i]].append(map[j])
                adj_list[map[j]].append(map[i])
                adj_list[map[i]].append(map[j])
                adj_list[map[j]].append(map[i])

        # test_labels[len(test_adjlists)] = int(label2index[label])
        test_labels[len(test_adjlists)] = int(label)
        test_adjlists.append(adj_list)
        test_features.append(torch.FloatTensor(feature).to(device))
        test_sequence.append(torch.tensor(smiles_seq))
        test_fg.append(fg)
    file.close()

    train_data = {}
    train_data['adj_lists'] = train_adjlists
    train_data['features'] = train_features
     # Pad train_sequence to length 100
    padded_train_sequence = []
    for seq in train_sequence:
      padded_seq = torch.nn.functional.pad(seq, (0, 100 - len(seq)), 'constant', 0)
      padded_train_sequence.append(padded_seq)
      train_data['sequence'] = padded_train_sequence
      train_data['fg'] = train_fg

    test_data = {}
    test_data['adj_lists'] = test_adjlists
    test_data['features'] = test_features
    padded_test_sequence = []
    for seq in test_sequence:
      padded_seq = torch.nn.functional.pad(seq, (0, 100 - len(seq)), 'constant', 0)
      padded_test_sequence.append(padded_seq)
      test_data['sequence'] = padded_test_sequence
      test_data['fg'] = test_fg   # 🔥 NEW

    return train_data, train_labels, test_data, test_labels

class CustomDataset(Dataset):
    def __init__(self, data_list, sequence_list, fg_list):
        self.data_list = data_list
        self.sequence_list = sequence_list
        self.fg_list = fg_list   # 🔥 NEW

    def __getitem__(self, index):
        data = self.data_list[index]
        sequence = self.sequence_list[index]
        fg = self.fg_list[index]   # 🔥 NEW

        return data, sequence, fg   # 🔥 UPDATED

    def __len__(self):
        return len(self.data_list)

def adj_list_to_adj_matrix(adj_list):
    num_nodes = max(adj_list.keys()) + 1
    adj_matrix = torch.zeros((num_nodes, num_nodes), dtype=torch.float)
    for node, neighbors in adj_list.items():
        for neighbor in neighbors:
            adj_matrix[node][neighbor] = 1.0
            adj_matrix[neighbor][node] = 1.0
    return adj_matrix

In [10]:
import torch
import torch.nn as nn
from torch_geometric.nn import TransformerConv

class GCNConvLayer(nn.Module):
    def __init__(self, in_channels, out_channels, heads, concat=True):
        super(GCNConvLayer, self).__init__()
        self.conv = TransformerConv(
            in_channels,
            out_channels,
            heads=heads,
            concat=concat
        )

    def forward(self, x, edge_index):
        return self.conv(x, edge_index)


class Graph_Transformer(nn.Module):
    def __init__(self, in_channels, hidden_channels, embed_dim, heads, dropout_rate=0.2):
        super(Graph_Transformer, self).__init__()

        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # First layer: keep concat=True
        self.conv1 = GCNConvLayer(
            in_channels,
            hidden_channels,
            heads,
            concat=True
        )

        # Second layer: 🔥 concat=False to control dimension
        self.conv2 = GCNConvLayer(
            hidden_channels * heads,
            embed_dim,
            heads=1,
            concat=False
        )

        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, data):
        x = data.x.to(self.device)
        edge_index = data.edge_index.to(self.device)

        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)

        # Global mean pooling (your version)
        x = x.mean(dim=0, keepdim=True)

        return x

In [11]:
import torch
import torch.nn as nn


class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_encoder_layers=3,
                 dim_feedforward=512, max_length=100, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_length, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers
        )

    def forward(self, sequence):
        if sequence.dim() == 1:
            sequence = sequence.unsqueeze(0)

        sequence = sequence.long()

        batch_size, seq_len = sequence.shape

        positions = torch.arange(seq_len, device=sequence.device)
        positions = positions.unsqueeze(0).expand(batch_size, seq_len)

        x = self.embedding(sequence) + self.pos_embedding(positions)

        padding_mask = sequence.eq(0)

        x = self.transformer_encoder(
            x,
            src_key_padding_mask=padding_mask
        )

        mask = (~padding_mask).float().unsqueeze(-1)
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)

        return x

In [12]:
from rdkit import Chem
import torch
import torch.nn as nn

FUNCTIONAL_GROUPS = {
    "alcohol": "[OX2H]",
    "phenol": "c[OX2H]",
    "amine_primary": "[NX3;H2]",
    "amine_secondary": "[NX3;H1]",
    "amine_tertiary": "[NX3;H0]",
    "amide": "C(=O)N",
    "carboxylic_acid": "C(=O)[OH]",
    "ester": "C(=O)O[#6]",
    "ether": "[OD2]([#6])[#6]",
    "aldehyde": "[CX3H1](=O)[#6]",
    "ketone": "[#6][CX3](=O)[#6]",
    "nitro": "[NX3](=O)=O",
    "halide": "[F,Cl,Br,I]",
    "thiol": "[SX2H]",
    "thioether": "[SX2]([#6])[#6]",
    "sulfonamide": "S(=O)(=O)N",
    "sulfone": "S(=O)(=O)[#6]",
    "phosphate": "P(=O)(O)(O)",
    "alkene": "C=C",
    "alkyne": "C#C",
    "aromatic": "a",
    "benzene_ring": "c1ccccc1",
    "nitrile": "C#N",
}

FG_PATTERNS = {
    name: Chem.MolFromSmarts(smarts)
    for name, smarts in FUNCTIONAL_GROUPS.items()
}

def extract_functional_groups(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return torch.zeros(len(FG_PATTERNS), dtype=torch.float)

    fg_vector = [
        int(mol.HasSubstructMatch(pattern))
        for pattern in FG_PATTERNS.values()
    ]

    return torch.tensor(fg_vector, dtype=torch.float)

In [13]:
class FunctionalGroupEmbedding(nn.Module):
    def __init__(self, fg_input_dim, embed_dim=128, dropout_rate=0.3):
        super(FunctionalGroupEmbedding, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(fg_input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(64, embed_dim),
            nn.ReLU()
        )

    def forward(self, fg):
        if fg.dim() == 1:
            fg = fg.unsqueeze(0)

        return self.fc(fg)

In [14]:
#Query---Graph  Key---SMILES+FG   Values---SMILES+FG
class CrossAttentionFusion(nn.Module):
    def __init__(self, embed_dim=128, num_heads=4, dropout=0.3):
        super().__init__()

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(embed_dim)

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, graph_emb, smiles_emb, fg_emb):
        """
        Query     = Graph
        Key/Value = SMILES + FG

        graph_emb  : [1,128]
        smiles_emb : [1,128]
        fg_emb     : [1,128]
        """

        # Query = graph embedding
        query = graph_emb.unsqueeze(1)        # [1,1,128]

        # Key/Value = SMILES + FG
        key_value = torch.stack(
            [smiles_emb.squeeze(0), fg_emb.squeeze(0)],
            dim=0
        ).unsqueeze(0)                        # [1,2,128]

        attn_output, attn_weights = self.cross_attention(
            query=query,
            key=key_value,
            value=key_value
        )

        fused = attn_output.squeeze(1)        # [1,128]

        # Residual connection with graph embedding
        fused = self.norm(fused + graph_emb)

        logits = self.classifier(fused)       # [1,1]

        return logits

In [14]:
#.................................................Query---smiles  Key---Graph+FG   Values---Graph+FG
class CrossAttentionFusion(nn.Module):
    def __init__(self, embed_dim=128, num_heads=4, dropout=0.3):
        super(CrossAttentionFusion, self).__init__()

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim)
        )

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, smiles_emb, graph_emb, fg_emb):
        """
        smiles_emb: [B, 128]
        graph_emb:  [B, 128]
        fg_emb:     [B, 128]
        """

        # Query = SMILES
        query = smiles_emb.unsqueeze(1)      # [B, 1, 128]

        # Key, Value = Graph + Functional Group
        key_value = torch.stack(
            [graph_emb, fg_emb],
            dim=1
        )                                   # [B, 2, 128]

        # Cross attention
        attn_output, attn_weights = self.cross_attention(
            query=query,
            key=key_value,
            value=key_value
        )

        # Residual + normalization
        fused = self.norm(query + attn_output)   # [B, 1, 128]

        # Feed-forward
        fused = fused + self.ffn(fused)          # [B, 1, 128]

        # Remove sequence dimension
        fused = fused.squeeze(1)                 # [B, 128]

        # Binary classification output
        logits = self.classifier(fused)          # [B, 1]
        return logits
        #return logits, fused, attn_weights

In [15]:
#Query---Graph  Key---SMILES+FG   Values---SMILES+FG
class FusionModel(nn.Module):
    def __init__(self, graph_model, sequence_model, fg_encoder, embed_dim=128):
        super().__init__()

        self.graph_model = graph_model
        self.sequence_model = sequence_model
        self.fg_encoder = fg_encoder

        self.cross_fusion = CrossAttentionFusion(
            embed_dim=embed_dim,
            num_heads=4,
            dropout=0.3
        )

    def forward(self, graph_data, sequence_inputs, fg):
        device = next(self.parameters()).device

        graph_data = graph_data.to(device)
        sequence_inputs = sequence_inputs.to(device)
        fg = fg.to(device)

        graph_emb = self.graph_model(graph_data)          # [1,128]
        smiles_emb = self.sequence_model(sequence_inputs) # [1,128]
        fg_emb = self.fg_encoder(fg)                      # [1,128]

        logits = self.cross_fusion(
            graph_emb,
            smiles_emb,
            fg_emb
        )

        return logits

In [15]:
#.......................................................Query---smiles  Key---Graph+FG   Values---Graph+FG
class FusionModel(nn.Module):
    def __init__(self, graph_model, sequence_model, fg_encoder, embed_dim=128):
        super().__init__()

        self.graph_model = graph_model
        self.sequence_model = sequence_model
        self.fg_encoder = fg_encoder

        self.cross_fusion = CrossAttentionFusion(
            embed_dim=embed_dim,
            num_heads=4,
            dropout=0.3
        )

    def forward(self, graph_data, sequence_inputs, fg):
        device = next(self.parameters()).device

        graph_data = graph_data.to(device)
        sequence_inputs = sequence_inputs.to(device)
        fg = fg.to(device)

        graph_emb = self.graph_model(graph_data)          # [1,128]
        smiles_emb = self.sequence_model(sequence_inputs) # [1,128]
        fg_emb = self.fg_encoder(fg)                      # [1,128]

        logits = self.cross_fusion(
            smiles_emb,
            graph_emb,
            fg_emb
        )                                                 # [1,1]

        return logits

In [16]:

#main
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
from datetime import datetime
from torch_geometric.data import Data
from torch_geometric.nn import TransformerConv
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score


def main(data_name,options):

    vocab_size = 100
    d_model = 128
    nhead = 4
    num_encoder_layers = 3
    dim_feedforward = 512
    max_length = 100
    batch_size = 1
    num_epochs = 100
    embed_dim = 128

    train_data, train_labels, test_data, test_labels = load_data_long(data_name, device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu"))
    device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    train_data, train_labels, test_data, test_labels = load_data_long(data_name, device)

    # convert train_labels to numpy
    train_labels_np = train_labels.cpu().numpy() if torch.is_tensor(train_labels) else train_labels
    test_labels_np = test_labels  # already numpy in your code

    print("Train:", np.bincount(train_labels_np.astype(int)))
    print("Test :", np.bincount(test_labels_np.astype(int)))
    input_dim_train = train_data['features'][0].size(-1)
    input_dim_test = test_data['features'][0].size(-1)


    adj_matrices_train = [adj_list_to_adj_matrix(adj_list) for adj_list in train_data['adj_lists']]
    adj_matrices_test = [adj_list_to_adj_matrix(adj_list) for adj_list in test_data['adj_lists']]


    data_list_train = [Data(x=features.clone().detach(),
                              edge_index=torch.nonzero(adj_matrix, as_tuple=False).t().contiguous(),
                              y=torch.tensor(label, dtype=torch.float))
                         for features, adj_matrix, label in zip(train_data['features'], adj_matrices_train, train_labels)]
    data_list_test = [Data(x=features.clone().detach(),
                                edge_index=torch.nonzero(adj_matrix, as_tuple=False).t().contiguous(),
                                y=torch.tensor(label, dtype=torch.float))
                            for features, adj_matrix, label in zip(test_data['features'], adj_matrices_test, test_labels)]

    train_dataset = CustomDataset(data_list_train, train_data['sequence'], train_data['fg'])
    test_dataset = CustomDataset(data_list_test, test_data['sequence'], test_data['fg'])

    if options[0]:
        graph_model = Graph_Transformer(in_channels=input_dim_train, hidden_channels=64, embed_dim=embed_dim, heads=4).to(device)
    if options[1]:
        sequence_model = TransformerModel(vocab_size, embed_dim, nhead, num_encoder_layers, dim_feedforward,max_length=100).to(device)
    if options[2]:
        fg_encoder = FunctionalGroupEmbedding(fg_input_dim=len(FUNCTIONAL_GROUPS), embed_dim=embed_dim).to(device)
    if options[3]:
        fusion_model = FusionModel(graph_model, sequence_model, fg_encoder, embed_dim=embed_dim).to(device)
        #fusion_model = FusionModel(graph_model, sequence_model).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(fusion_model.parameters(), lr=0.0001, weight_decay=5e-4)


    current_datetime = datetime.now()
    formatted_datetime = current_datetime.strftime('%Y-%m-%d_%H-%M-%S')
    session_name = f'{data_name}_{formatted_datetime}'
    folder_path = os.path.join('saved_models', session_name)
    os.makedirs(folder_path, exist_ok=True)

    output_dir_train = f'output/{data_name}/train'
    os.makedirs(output_dir_train, exist_ok=True)
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    file_name_train = f'{output_dir_train}/train_accuracy_details_{current_time}.txt'

    output_dir_test = f'output/{data_name}/test'
    os.makedirs(output_dir_test, exist_ok=True)
    current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    file_name_test = f'{output_dir_test}/test_accuracy_details_{current_time}.txt'


    best_train_accuracy = 0.0
    best_test_accuracy=0.0
    # Training loop

    with open(file_name_train, 'a') as file_train, open(file_name_test, 'a') as file_test:
        for epoch in range(100):
            total_correct = 0
            total_samples = 0
            true_labels_train = []
            pred_probs_train = []
            losses=0.0

            for data_batch in train_dataset:
                graph_data_batch = data_batch[0]
                sequence_inputs  = data_batch[1]
                fg = data_batch[2]
                if len(fg.shape) == 1:
                  fg = fg.unsqueeze(0)
                fg = fg.to(device)
                sequence_targets=graph_data_batch.y

                # Zero the gradients
                optimizer.zero_grad()

                # Forward pass
                output= fusion_model(graph_data_batch, sequence_inputs,fg)

                # Compute binary predictions
                prob = torch.sigmoid(output)#..............................
                binary_predictions = (prob >= 0.5).float()#...................
                #binary_predictions = (output >= 0.5).float()

                # Move and reshape sequence_targets BEFORE comparison
                sequence_targets = sequence_targets.view(-1, 1).to(device)

                # Compute batch accuracy
                batch_correct = (binary_predictions == sequence_targets).sum().item()
                total_samples += 1 #total_samples should be 1 since batch_size is 1
                total_correct += batch_correct # Add batch_correct after reshape


                # output is already on device, sequence_targets is now correctly shaped and on device
                # The following lines are either redundant or were incorrectly placed
                # output = output.to(device)
                # sequence_targets = sequence_targets.view(-1, 1).to(device)

                true_labels_train.append(sequence_targets.cpu().numpy().reshape(-1))
                pred_probs_train.append(prob.detach().cpu().numpy()[0]) # Ensure 1D array...............
                #print(output,sequence_targets,pred_probs_train)

                # Cast sequence_inputs to float
                # sequence_inputs = sequence_inputs.float() # This line is not needed here anymore, it will be handled inside FusionModel for the correct tensor


                # Compute loss

                loss = criterion(output, sequence_targets)
                losses+=loss.item()

                # Backward pass
                loss.backward()

                # Update weights
                optimizer.step()


            # Compute epoch accuracy
            epoch_train_accuracy = (total_correct / total_samples)*100
            print(f"Epoch {epoch+1}/{100}, Epoch Accuracy: {epoch_train_accuracy:.4f}")

            if epoch_train_accuracy >= best_train_accuracy:
                best_train_accuracy = epoch_train_accuracy
                model_path = os.path.join(folder_path, f'train_best_model_{best_train_accuracy:.3f}.pth')
                torch.save(fusion_model.state_dict(), model_path)
                print("Saved model with accuracy train model with accuracy{:.2f}% to {}".format(best_train_accuracy, model_path))

            true_labels_train = np.concatenate(true_labels_train).astype(int) # Convert to int
            pred_probs_train = np.nan_to_num(np.concatenate(pred_probs_train), nan=0)
            predicted_labels_train_int = (pred_probs_train >= 0.5).astype(int)

            precision_train = precision_score(true_labels_train, predicted_labels_train_int)
            recall_train = recall_score(true_labels_train, predicted_labels_train_int)
            auc_roc_train = roc_auc_score(true_labels_train, pred_probs_train)
            f1_train = f1_score(true_labels_train, predicted_labels_train_int)
            print(f"Train AUC-ROC: {auc_roc_train:.4f}, Train F1 Score: {f1_train:.4f} , Train Precision: {precision_train:.4f}, Train Recall: {recall_train:.4f}\n")
            file_train.write(f'Epoch {epoch + 1}/{num_epochs}, Train Loss: {losses:.4f}, Train Accuracy: {epoch_train_accuracy:.4f}, Train AUC-ROC: {auc_roc_train:.4f}, Train F1 Score: {f1_train:.4f} , Train Precision: {precision_train:.4f}, Train Recall: {recall_train:.4f}\n')

            total_correct = 0
            total_samples = 0
            true_labels_test = []
            pred_probs_test = []

            for data_batch in test_dataset:
                graph_data_batch = data_batch[0]
                sequence_inputs = data_batch[1]
                fg = data_batch[2]
                if len(fg.shape) == 1:
                  fg = fg.unsqueeze(0)
                fg = fg.to(device)
                sequence_targets = graph_data_batch.y


                output= fusion_model(graph_data_batch, sequence_inputs,fg)
                prob = torch.sigmoid(output)
                binary_predictions = (prob >= 0.5).float()#......................
                #binary_predictions = (output >= 0.5).float()

                # Move and reshape sequence_targets BEFORE comparison
                sequence_targets = sequence_targets.view(-1, 1).to(device)

                batch_correct = (binary_predictions == sequence_targets).sum().item()
                total_samples += 1 #total_samples should be 1 since batch_size is 1
                total_correct += batch_correct

                true_labels_test.append(sequence_targets.cpu().numpy().reshape(-1))
                pred_probs_test.append(prob.detach().cpu().numpy()[0])#...............

            epoch_test_accuracy = (total_correct / total_samples)*100
            print(f"Epoch Testing Accuracy : {epoch_test_accuracy:.4f}")

            if epoch_test_accuracy >= best_test_accuracy:
                best_test_accuracy = epoch_test_accuracy
                model_path = os.path.join(folder_path, f'test_best_model_{best_test_accuracy:.3f}.pth')
                torch.save(fusion_model.state_dict(), model_path)
                print("Saved model with Test Model with accuracy {:.2f}% to {}".format(best_test_accuracy, model_path))


            true_labels_test = np.concatenate(true_labels_test).astype(int) # Convert to int
            pred_probs_test = np.nan_to_num(np.concatenate(pred_probs_test), nan=0)
            predicted_labels_test_int = (pred_probs_test >= 0.5).astype(int)

            print("Prob min:", pred_probs_test.min())
            print("Prob max:", pred_probs_test.max())
            print("Prob mean:", pred_probs_test.mean())
            print("Pred counts:", np.bincount(predicted_labels_test_int))

            precision_test = precision_score(true_labels_test, predicted_labels_test_int)
            recall_test = recall_score(true_labels_test, predicted_labels_test_int)
            auc_roc_test = roc_auc_score(true_labels_test, pred_probs_test)
            f1_test = f1_score(true_labels_test, predicted_labels_test_int)
            print(f"Test AUC-ROC: {auc_roc_test:.4f}, Test F1 Score: {f1_test:.4f}, Test Precision: {precision_test:.4f}, Test Recall: {recall_test:.4f}\n")
            file_test.write(f'Epoch {epoch + 1}/{num_epochs}, Test Accuracy: {epoch_test_accuracy:.4f},Test AUC-ROC: {auc_roc_test:.4f}, Test F1 Score: {f1_test:.4f}, Test Precision: {precision_test:.4f}, Test Recall: {recall_test:.4f} \n')
    file_test.close()
    file_train.close()




if __name__ == "__main__":
    options=[True,True,True,True]
    data_name='bace'
    main(data_name,options)


Train: [637 573]
Test : [91 61]


/tmp/ipykernel_23598/74051640.py:46: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y=torch.tensor(label, dtype=torch.float))


Epoch 1/100, Epoch Accuracy: 52.5620
Saved model with accuracy train model with accuracy52.56% to saved_models/bace_2026-05-05_07-02-48/train_best_model_52.562.pth
Train AUC-ROC: 0.5273, Train F1 Score: 0.4595 , Train Precision: 0.4990, Train Recall: 0.4258

Epoch Testing Accuracy : 60.5263
Saved model with Test Model with accuracy 60.53% to saved_models/bace_2026-05-05_07-02-48/test_best_model_60.526.pth
Prob min: 0.37655866
Prob max: 0.51358277
Prob mean: 0.42783284
Pred counts: [149   3]
Test AUC-ROC: 0.4995, Test F1 Score: 0.0625, Test Precision: 0.6667, Test Recall: 0.0328

Epoch 2/100, Epoch Accuracy: 51.5702
Train AUC-ROC: 0.5176, Train F1 Score: 0.3832 , Train Precision: 0.4828, Train Recall: 0.3176

Epoch Testing Accuracy : 59.2105
Prob min: 0.35528725
Prob max: 0.5084924
Prob mean: 0.40857908
Pred counts: [151   1]
Test AUC-ROC: 0.4947, Test F1 Score: 0.0000, Test Precision: 0.0000, Test Recall: 0.0000

Epoch 3/100, Epoch Accuracy: 48.9256
Train AUC-ROC: 0.4907, Train F1 Scor

KeyboardInterrupt: 